# Figure S1: Benchmark Extension

- Panels: `FigS1a`, `FigS1b`, `FigS1c`, `FigS1d`
- Role: condition-level distribution extension for the main benchmark
- Inputs: `benchmark_summary_all.csv` + per-model `metrics.csv`
- Notes: Norman remains `single-only` to match the main benchmark definition


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
apply_gears_paper_style(font_scale=1.0)

DATASET_ORDER = ['Adamson', 'Norman', 'Dixit']
MODEL_ORDER = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt']
MODEL_LABELS = {'trishift_nearest':'TriShift','gears':'GEARS','biolord':'biolord','genepert':'GenePert','scgpt':'scGPT'}
MODEL_COLORS = {'TriShift':'#9FD9D3','GEARS':'#F2B56B','biolord':'#F0806A','GenePert':'#87A8D8','scGPT':'#DDD3C8'}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS1_BenchmarkExtension'
OUT_ROOT.mkdir(parents=True, exist_ok=True)


def omit_scouter_style_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    work = df.copy()
    dataset_col = 'dataset' if 'dataset' in work.columns else ('dataset_key' if 'dataset_key' in work.columns else None)
    model_col = 'model_name' if 'model_name' in work.columns else ('method' if 'method' in work.columns else None)
    if dataset_col is None or model_col is None:
        return work
    drop_mask = (
        work[dataset_col].astype(str).str.lower().eq('dixit')
        & work[model_col].astype(str).str.lower().eq('biolord')
    )
    return work.loc[~drop_mask].copy()

def _style_axis(ax):
    paper_style_axis(ax, grid_axis='y')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)



In [2]:
summary_path = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig2_MultiDatasetBenchmark' / 'benchmark_summary_all.csv'
summary_df = pd.read_csv(summary_path)
summary_df = omit_scouter_style_rows(summary_df)
summary_df = summary_df[summary_df['model_name'].isin(MODEL_ORDER)].copy()
summary_df['dataset_label'] = pd.Categorical(summary_df['dataset_label'], DATASET_ORDER, ordered=True)
summary_df['display_name'] = summary_df['model_name'].map(MODEL_LABELS)

detail_rows = []
for row in summary_df.itertuples(index=False):
    metrics_path = Path(row.metrics_path)
    if not metrics_path.exists():
        continue
    metrics_df = pd.read_csv(metrics_path)
    if str(row.dataset).strip().lower() == 'norman' and 'subgroup' in metrics_df.columns:
        metrics_df = metrics_df[metrics_df['subgroup'].astype(str).str.lower() == 'single'].copy()
    keep_cols = [c for c in ['condition','pearson','nmse','deg_mean_r2','systema_corr_20de_allpert'] if c in metrics_df.columns]
    if not keep_cols:
        continue
    keep = metrics_df[keep_cols].copy()
    keep['dataset'] = row.dataset
    keep['dataset_label'] = row.dataset_label
    keep['model_name'] = row.model_name
    keep['display_name'] = row.display_name
    detail_rows.append(keep)
detail_df = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
detail_df = omit_scouter_style_rows(detail_df)
detail_df['dataset_label'] = pd.Categorical(detail_df['dataset_label'], DATASET_ORDER, ordered=True)
display(detail_df.head())


,condition,pearson,nmse,deg_mean_r2,systema_corr_20de_allpert,dataset,dataset_label,model_name,display_name
0,TELO2+ctrl,0.883029,0.143366,0.723269,0.415978,adamson,Adamson,trishift_nearest,TriShift
1,MANF+ctrl,0.998023,0.007192,0.991113,0.976758,adamson,Adamson,trishift_nearest,TriShift
2,ATP5B+ctrl,0.989362,0.042449,0.956899,0.795275,adamson,Adamson,trishift_nearest,TriShift
3,MRPL39+ctrl,0.934642,0.119983,0.872170,0.114861,adamson,Adamson,trishift_nearest,TriShift
4,DERL2+ctrl,0.856557,0.134916,0.709364,-0.849630,adamson,Adamson,trishift_nearest,TriShift


In [3]:
metric_specs = [
    ('pearson', 'Pearson', 'figs1a_pearson_boxplot.png'),
    ('nmse', 'nMSE', 'figs1b_nmse_boxplot.png'),
    ('deg_mean_r2', r'DEG mean $R^2$', 'figs1c_deg_mean_r2_boxplot.png'),
    ('systema_corr_20de_allpert', 'Systema Pearson', 'figs1d_systema_corr_boxplot.png'),
]

for metric_col, ylabel, out_name in metric_specs:
    plot_df = detail_df.dropna(subset=[metric_col]).copy()
    if plot_df.empty:
        continue
    fig, ax = plt.subplots(figsize=(7.4, 5.1), dpi=220)
    dataset_order = [label for label in DATASET_ORDER if label in set(plot_df['dataset_label'].astype(str))]
    label_order = [MODEL_LABELS[m] for m in MODEL_ORDER if MODEL_LABELS[m] in set(plot_df['display_name'].astype(str))]
    seen_labels = []
    group_width = 0.82
    for dataset_idx, dataset_label in enumerate(dataset_order):
        sub = plot_df[plot_df['dataset_label'].astype(str) == str(dataset_label)].copy()
        present_labels = [label for label in label_order if label in set(sub['display_name'].astype(str))]
        if not present_labels:
            continue
        base_width = min(0.16, group_width / max(len(present_labels), 1))
        box_width = base_width * 0.9
        if len(present_labels) == 1:
            offsets = [0.0]
        else:
            offsets = np.linspace(-base_width * (len(present_labels) - 1) / 2, base_width * (len(present_labels) - 1) / 2, len(present_labels))
        for offset, label in zip(offsets, present_labels):
            vals = sub.loc[sub['display_name'].astype(str) == str(label), metric_col].astype(float).dropna().to_numpy()
            if vals.size == 0:
                continue
            bp = ax.boxplot(
                [vals],
                positions=[dataset_idx + float(offset)],
                widths=box_width,
                patch_artist=True,
                whis=1.5,
                showfliers=True,
                boxprops={'facecolor': MODEL_COLORS[label], 'edgecolor': 'black', 'linewidth': 0.8},
                medianprops={'color': 'black', 'linewidth': 1.0},
                whiskerprops={'color': 'black', 'linewidth': 0.8},
                capprops={'color': 'black', 'linewidth': 0.8},
                flierprops={'marker': 'o', 'markersize': 2.0, 'markerfacecolor': '#666666', 'markeredgecolor': '#666666', 'alpha': 0.8},
            )
            if label not in seen_labels:
                seen_labels.append(label)
    ax.set_xticks(np.arange(len(dataset_order)))
    ax.set_xticklabels(dataset_order)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, pad=8)
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=MODEL_COLORS[label], edgecolor='black', linewidth=0.8) for label in label_order]
    ax.legend(
        handles,
        label_order,
        title='',
        frameon=False,
        ncol=5,
        loc='upper center',
        bbox_to_anchor=(0.5, 1.22),
        borderaxespad=0.0,
        handlelength=1.6,
        columnspacing=1.4,
    )
    _style_axis(ax)
    fig.tight_layout(rect=[0, 0, 1, 0.86])
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plot_df.to_csv(OUT_ROOT / out_name.replace('.png', '_values.csv'), index=False)
    plt.close(fig)
